In [ ]:
image_folder = r"/home/anvy4548/projects/crystal-recognition/test_images/different_contrast_test/images"
mask_folder = r"/home/anvy4548/projects/crystal-recognition/test_images/different_contrast_test/masks"

In [3]:
import cv2
import os
import numpy as np
from tqdm import tqdm
import itertools
import time


def adjust_brightness_contrast(image, brightness=0, contrast=1.0):
    return cv2.convertScaleAbs(image, alpha=contrast, beta=brightness)


def add_poisson_noise(image, scale=1.0):
    """
    Poisson noise.
    Lower scale → stronger noise.
    scale = 0 → no noise
    """
    if scale == 0:
        return image

    image_float = image.astype(np.float32) / 255.0
    noisy = np.random.poisson(image_float * scale) / scale
    noisy = np.clip(noisy, 0, 1.0)

    return (noisy * 255).astype(np.uint8)


def process_mask(mask):
    # invert
    mask = cv2.bitwise_not(mask)

    # force strict binary 0/255
    mask = np.where(mask > 0, 255, 0).astype(np.uint8)

    return mask


def systematic_intensity_variation(
    images_directory,
    masks_directory,
    output_images_directory,
    output_masks_directory,
    brightness_values=(-40, 0, 40),
    contrast_values=(0.7, 1.0, 1.3),
    poisson_scales=(0, 0.5, 1.0, 2.0)
):

    os.makedirs(output_images_directory, exist_ok=True)
    os.makedirs(output_masks_directory, exist_ok=True)

    valid_exts = (".png", ".tif", ".tiff")

    image_files = [
        f for f in os.listdir(images_directory)
        if f.lower().endswith(valid_exts)
    ]

    combinations = list(itertools.product(
        brightness_values,
        contrast_values,
        poisson_scales
    ))

    start = time.time()
    total_saved = 0

    for image_file in tqdm(image_files, desc="Systematic intensity testing"):

        image_path = os.path.join(images_directory, image_file)
        mask_path = os.path.join(masks_directory, image_file)

        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if image is None or mask is None:
            continue

        mask = process_mask(mask)

        base_name, ext = os.path.splitext(image_file)

        for b, c, p in combinations:

            out_img = adjust_brightness_contrast(image, brightness=b, contrast=c)

            if p != 0:
                out_img = add_poisson_noise(out_img, scale=p)

            filename = f"{base_name}_b{b}_c{c}_p{p}{ext}"

            cv2.imwrite(os.path.join(output_images_directory, filename), out_img)
            cv2.imwrite(os.path.join(output_masks_directory, filename), mask)

            total_saved += 1

    elapsed = time.time() - start
    print(f"Created {total_saved} image/mask pairs in {round(elapsed, 2)} s")

    return {
        "input_images": len(image_files),
        "variations_per_image": len(combinations),
        "total_generated": total_saved,
        "time_spent_s": round(elapsed, 2),
    }

In [2]:
brightness_values = (-40, 0, 40)
contrast_values   = (0.7, 1.0, 1.3)
noise_sigmas      = (0, 5, 15)

In [5]:
input_directory = r"/home/anvy4548/projects/crystal-recognition/test_images/different_contrast_test/images"
output_directory = r"/home/anvy4548/projects/crystal-recognition/test_images/different_contrast_test/aug"

images_directory="/home/anvy4548/projects/crystal-recognition/test_images/different_contrast_test/images"
masks_directory="/home/anvy4548/projects/crystal-recognition/test_images/different_contrast_test/masks"
output_images_directory="/home/anvy4548/projects/crystal-recognition/test_images/different_contrast_test/aug/images"
output_masks_directory="/home/anvy4548/projects/crystal-recognition/test_images/different_contrast_test/aug/masks"

systematic_intensity_variation(images_directory, masks_directory, output_images_directory, output_masks_directory)

Systematic intensity testing: 100%|██████████| 4/4 [00:01<00:00,  3.28it/s]

Created 144 image/mask pairs in 1.22 s


{'input_images': 4,
 'variations_per_image': 36,
 'total_generated': 144,
 'time_spent_s': 1.22}